# Limpieza básica de los GRD respiratorios (2019-2024)

Este notebook toma los `GRD_PUBLICO_AAAA_ER.txt` (ya filtrados por `DIAGNOSTICO1` que empieza con `J`) y deja en memoria un GRD limpio por año.

**Qué hace, en orden**

1. Lectura con la codificación correcta de cada año y limpieza de texto (espacios, vacíos).
2. Unificación de nombres de columnas entre años (`CIP_ENCRIPTADO` / `ID_BENEFICIARIO`).
3. **Duplicados** (con justificación).
4. **Nulos y valores "falsos nulos"** (con justificación).
5. Fechas a un único tipo `datetime64` (se muestran como `dd/mm/yyyy`).
6. Variables del proyecto: `SEXO`, `EDAD`, `TRAMO_EDAD`, `FALLECIDO`, `DIAGNOSTICO1` (categoría CIE-10 de 3 caracteres), `FECHA_INGRESO`, `FECHA_ALTA`, `MES_ALTA`, `CUT`, `HOSPITAL`, `COMUNA` (residencia) y `REGION`.
7. Reporte de calidad (antes/después) y verificación.

> **Fecha de referencia: fecha de alta.** Cada archivo `GRD_PUBLICO_AAAA` contiene exactamente los egresos con fecha de alta en ese año, por lo que los 72 meses de 2019-2024 quedan completos. Con la fecha de ingreso los bordes serían parciales (ingresos de 2018 y diciembre de 2024 incompletos). `FECHA_INGRESO` se conserva para detectar duplicados y calcular la estadía.

> `REGION` se obtiene de `catalogo_comuna_region.csv` (una región por comuna de residencia), usado por el mapa del frontend.

In [8]:
import re
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------- Configuración ----------------
DIR_GRD     = Path('../GRD_CIE_TM')

ANIOS = range(2019, 2025)

# Codificación real de cada archivo _ER.txt (verificada: 2019-2024 NO son utf-8 válido; 2025 sí lo es).
ENCODING = {anio: 'latin-1' for anio in ANIOS}

# Si True, FECHA_INGRESO se guarda como texto 'dd/mm/yyyy'.
# Recomendado False: en parquet se guardan como fecha real (se pueden filtrar y ordenar) y
# el formato dd/mm/yyyy se aplica solo al mostrarlas (ver última celda).
FECHAS_COMO_TEXTO = False

## 1. Catálogos (CIE-10, tabla maestra de hospitales y región)


In [9]:
def limpiar_texto(s):
    """strip + vacíos -> NA"""
    s = s.astype('string').str.strip()
    return s.mask(s == '')

# --- CIE-10 (diagnósticos) ---
cie10 = pd.read_excel(DIR_GRD / 'CIE-10.xlsx', dtype=str)
cie10['Código'] = limpiar_texto(cie10['Código']).str.upper()
DESC_CIE10 = cie10.drop_duplicates('Código').set_index('Código')['Descripción']

# --- Tabla maestra de hospitales ---
maestra = pd.read_excel(DIR_GRD / 'Tablas maestras bases GRD .xlsx', sheet_name='Hospitales')
maestra.columns = ['COD_HOSPITAL', 'HOSPITAL']
maestra['COD_HOSPITAL'] = limpiar_texto(maestra['COD_HOSPITAL'])
maestra['HOSPITAL'] = limpiar_texto(maestra['HOSPITAL'])
NOMBRE_HOSPITAL = maestra.drop_duplicates('COD_HOSPITAL').set_index('COD_HOSPITAL')['HOSPITAL']

# --- Región por comuna (catálogo versionado en el repositorio) ---
RUTA_CATALOGO = Path('catalogo_comuna_region.csv')
catalogo = pd.read_csv(RUTA_CATALOGO, dtype=str, encoding='utf-8-sig')
catalogo['COMUNA'] = limpiar_texto(catalogo['COMUNA']).str.upper()
assert catalogo['COMUNA'].is_unique, 'Hay comunas repetidas en el catálogo'
REGION_POR_COMUNA = catalogo.set_index('COMUNA')['REGION'].to_dict()
CUT_POR_COMUNA = catalogo.set_index('COMUNA')['CUT'].to_dict()
COD_REGION_POR_COMUNA = catalogo.set_index('COMUNA')['COD_REGION'].to_dict()

## 2. Funciones de limpieza

### Decisiones y justificación

**Duplicados**
* Se **eliminan solo los duplicados exactos** (las 129 columnas idénticas). Un registro idéntico en todo, incluyendo fechas de traslado, alta, GRD y médico encriptado, es un error de extracción, no dos hospitalizaciones reales. Esto afecta casi solo a 2020 (~128 filas) y 2022 (2 filas).
* Se consideran **posibles duplicados** los registros que coinciden en paciente, hospital, fecha de ingreso, fecha de alta y categoría diagnóstica (3 caracteres): el mismo egreso registrado dos veces. Se conserva uno y se elimina el resto. Si la fecha de alta difiere, se consideran episodios distintos y se conservan. El total eliminado queda en el reporte de calidad.

**Nulos**
* **Falsos nulos de texto**: `DESCONOCIDO`/`DESCONOCIDA`/`NO APLICA` dentro de columnas de fecha se convierten en nulo real.
* **Nulos estructurales** (`DIAGNOSTICO2-35`, `FECHATRASLADO*`, `*RN*`): el vacío significa "no aplica" (el paciente no tuvo ese diagnóstico secundario o traslado). **No se imputan ni se eliminan filas.**
* **Falsos nulos** (`DESCONOCIDO`, `NO APLICA` dentro de columnas de fecha): se convierten a nulo real (`NaT`), porque una fecha no puede ser "desconocido".
* **`SEXO = DESCONOCIDO`** (<0.02% de las filas): se conserva como categoría explícita.
* Edad: se calcula en años cumplidos al ingreso; fuera de 0–109 queda nula y su tramo como DESCONOCIDO. La fecha de nacimiento se descarta después de calcularla/
* Las filas sin FECHAALTA o sin FECHA_INGRESO se eliminan (5 filas en 2024), porque MES_ALTA y la regla de duplicados dependen de ellas

In [10]:
FALSOS_NULOS = {'DESCONOCIDO', 'DESCONOCIDA', 'NO APLICA', 'NO INFORMADO', 'S/I', 'SIN INFORMACION', 'SIN INFORMACIÓN'}

COLS_NUM_COMA = ['IR_29301_PESO', 'PESORN1', 'PESORN2', 'PESORN3', 'PESORN4']
COLS_NUM = ['RN1ESTADO', 'RN2ESTADO', 'RN3ESTADO', 'RN4ESTADO', 'USOSPABELLON',
            'IR_29301_SEVERIDAD', 'IR_29301_MORTALIDAD']

def parsear_fecha(serie):
    """Acepta 'yyyy-mm-dd' y 'dd-mm-yyyy' (formato por año/columna distinto). Lo demás -> NaT."""
    s = limpiar_texto(serie)
    s = s.mask(s.str.upper().isin(FALSOS_NULOS))
    es_iso = s.str.match(r'^\d{4}-\d{2}-\d{2}').fillna(False).astype(bool)
    es_dmy = s.str.match(r'^\d{2}-\d{2}-\d{4}').fillna(False).astype(bool)
    f_iso = pd.to_datetime(s.where(es_iso).str[:10], format='%Y-%m-%d', errors='coerce')
    f_dmy = pd.to_datetime(s.where(es_dmy).str[:10], format='%d-%m-%Y', errors='coerce')
    return f_iso.fillna(f_dmy)

# Tramos de edad para estandarizar tasas (EDAD se conserva para filtros con otros tramos).
BINS_TRAMO = [0, 5, 13, 18, 30, 50, 65, 110]
ETIQUETAS_TRAMO = ['Infancia (0-4)', 'Niñez (5-12)', 'Adolescencia (13-17)', 'Adultos jóvenes (18-29)',
                   'Adultos (30-49)', 'Adultos maduros (50-64)', 'Adultos mayores (65+)']

def calcular_edad(f_nac, f_ref):
    """Edad en años cumplidos a la fecha de referencia. Fuera de 0-109 -> nulo."""
    edad = f_ref.dt.year - f_nac.dt.year
    aun_no_cumple = ((f_ref.dt.month < f_nac.dt.month) |
                     ((f_ref.dt.month == f_nac.dt.month) & (f_ref.dt.day < f_nac.dt.day)))
    edad = edad - aun_no_cumple.astype(int)
    return edad.where((edad >= 0) & (edad <= 109)).astype('Int64')

def limpiar_grd(anio):
    ruta = DIR_GRD / f'GRD_PUBLICO_{anio}_ER.txt'
    df = pd.read_csv(ruta, sep='|', encoding=ENCODING[anio], dtype=str)
    rep = {'anio': anio, 'filas_originales': len(df)}

    # --- 1. Texto: strip y vacíos -> NA ---
    for c in df.columns:
        df[c] = limpiar_texto(df[c])

    # --- 2. Unificar nombre del ID (2019-2023: CIP_ENCRIPTADO; 2024-2025: ID_BENEFICIARIO) ---
    df = df.rename(columns={'CIP_ENCRIPTADO': 'ID_BENEFICIARIO', 'FECHAALTA': 'FECHA_ALTA'})

    # --- 3. Duplicados exactos ---
    n = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    rep['duplicados_exactos_eliminados'] = n - len(df)

    # --- 4. Fechas (todas las columnas FECHA*) -> datetime64 ---
    cols_fecha = [c for c in df.columns if c.startswith('FECHA')]
    for c in cols_fecha:
        crudo = df[c]
        df[c] = parsear_fecha(crudo).astype('datetime64[ns]')     # misma unidad en todos los años
        malas = crudo.notna() & df[c].isna() & ~crudo.str.upper().isin(FALSOS_NULOS)
        if malas.any():
            rep[f'fecha_no_parseable_{c}'] = int(malas.sum())

    # Fecha de referencia del proyecto: FECHA DE ALTA
    # FECHA_INGRESO se conserva para duplicados y estadía.
    rep['FECHA_ALTA_nula'] = int(df['FECHA_ALTA'].isna().sum())
    rep['FECHA_INGRESO_nula'] = int(df['FECHA_INGRESO'].isna().sum())
    df = df.dropna(subset=['FECHA_ALTA', 'FECHA_INGRESO']).reset_index(drop=True)
    df['MES_ALTA'] = df['FECHA_ALTA'].dt.month.astype('Int64')
    rep['FECHA_ALTA_fuera_del_anio'] = int((df['FECHA_ALTA'].dt.year != anio).sum())
    rep['alta_antes_que_ingreso'] = int((df['FECHA_ALTA'] < df['FECHA_INGRESO']).sum())

    # --- 5. Numéricos (coma decimal, ceros a la izquierda del código GRD) ---
    for c in COLS_NUM_COMA:
        if c in df: df[c] = pd.to_numeric(df[c].str.replace(',', '.', regex=False), errors='coerce')
    for c in COLS_NUM:
        if c in df: df[c] = pd.to_numeric(df[c], errors='coerce')
    df['IR_29301_COD_GRD'] = df['IR_29301_COD_GRD'].str.lstrip('0')   # '044163' (2019-23) vs '41203' (2025)

    # --- 6. ANIO ---
    df.insert(0, 'ANIO', anio)

    # --- 7. SEXO ---
    df['SEXO'] = df['SEXO'].str.upper()
    df.loc[~df['SEXO'].isin(['HOMBRE', 'MUJER']), 'SEXO'] = 'DESCONOCIDO'

    # --- 7b. FALLECIDO (binario) a partir del tipo de alta ---
    if 'TIPOALTA' in df:
        df['FALLECIDO'] = df['TIPOALTA'].str.upper().str.contains('FALLEC', na=False).astype(int)
    else:
        df['FALLECIDO'] = pd.NA
    rep['FALLECIDO_sin_dato'] = int(df['FALLECIDO'].isna().sum())

    # --- 8. Diagnóstico principal: categoría CIE-10 (3 caracteres) y su descripción ---
    df['DIAGNOSTICO1'] = df['DIAGNOSTICO1'].str.upper().str[:3]
    df['DIAGNOSTICO1_DESC'] = df['DIAGNOSTICO1'].map(DESC_CIE10)
    rep['DIAGNOSTICO1_sin_descripcion'] = int(df['DIAGNOSTICO1_DESC'].isna().sum())

    # Procedimientos CIE-9: ya no se usan -> se descartan las 30 columnas
    df = df.drop(columns=[f'PROCEDIMIENTO{i}' for i in range(1, 31)], errors='ignore')

    # --- 9. Posible duplicado: el mismo egreso registrado dos veces ---
    # Mismo paciente, hospital, fecha de ingreso, fecha de alta y categoría diagnóstica.
    # Si la fecha de alta difiere se consideran episodios distintos y se conservan.
    clave = ['ID_BENEFICIARIO', 'COD_HOSPITAL', 'FECHA_INGRESO', 'FECHA_ALTA', 'DIAGNOSTICO1']
    df['POSIBLE_DUPLICADO'] = df.duplicated(clave, keep='first') & df['ID_BENEFICIARIO'].notna()
    rep['posibles_duplicados_marcados'] = int(df['POSIBLE_DUPLICADO'].sum())
    df = df.loc[~df['POSIBLE_DUPLICADO']].reset_index(drop=True)
    rep['posibles_duplicados_eliminados'] = rep['posibles_duplicados_marcados']

    # --- 10. Edad al ingreso y tramo de edad; luego se descarta la fecha de nacimiento ---
    df['EDAD'] = calcular_edad(df['FECHA_NACIMIENTO'], df['FECHA_INGRESO'])
    df['TRAMO_EDAD'] = (pd.cut(df['EDAD'].astype('float'), bins=BINS_TRAMO, right=False, labels=ETIQUETAS_TRAMO)
                          .astype('string').fillna('DESCONOCIDO'))
    rep['EDAD_nula'] = int(df['EDAD'].isna().sum())
    df = df.drop(columns=['FECHA_NACIMIENTO', 'PROVINCIA'], errors='ignore')

    # --- 11. Hospital, comuna y región---
    df['COD_HOSPITAL'] = df['COD_HOSPITAL'].str.strip()
    df['HOSPITAL'] = df['COD_HOSPITAL'].map(NOMBRE_HOSPITAL)
    rep['hospital_sin_maestra'] = int(df['HOSPITAL'].isna().sum())
    rep['hospital_sin_maestra_cods'] = ', '.join(df.loc[df['HOSPITAL'].isna(), 'COD_HOSPITAL'].dropna().unique())
    df['HOSPITAL'] = df['HOSPITAL'].fillna('SIN CATALOGAR (' + df['COD_HOSPITAL'] + ')')

    # Códigos oficiales: CUT comunal y código de región.
    df['CUT'] = df['COMUNA'].map(CUT_POR_COMUNA)
    df['COD_REGION'] = df['COMUNA'].map(COD_REGION_POR_COMUNA)

        # COMUNA = comuna de RESIDENCIA del paciente (columna original del GRD).
    df['COMUNA'] = df['COMUNA'].str.upper()
    rep['COMUNA_nula'] = int(df['COMUNA'].isna().sum())

    # REGION desde la COMUNA: cada comuna pertenece a una sola región (necesario para el mapa por comunas).
    df['REGION'] = df['COMUNA'].map(REGION_POR_COMUNA)
    rep['REGION_nula'] = int(df['REGION'].isna().sum())

    # --- 12. Orden: variables clave primero ---
    clave_primero = ['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'REGION', 'CUT', 'COD_REGION', 'ID_BENEFICIARIO',
                     'SEXO', 'EDAD', 'TRAMO_EDAD', 'FALLECIDO', 'FECHA_INGRESO', 'FECHA_ALTA', 'MES_ALTA',
                     'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC', 'POSIBLE_DUPLICADO']
    resto = [c for c in df.columns if c not in clave_primero]
    df = df[clave_primero + resto]

    rep['filas_finales'] = len(df)
    return df, rep

## 3. Ejecutar todos los años y exportar

Se procesa un año a la vez (y se libera memoria) porque los archivos son grandes. Con `FECHAS_COMO_TEXTO = True` las fechas se guardan como texto `dd/mm/yyyy`.

In [11]:
reportes = []
grd = {}   # solo las columnas de interés, en memoria (nada se escribe a disco)

COLS_VER = ['ANIO', 'COD_HOSPITAL', 'HOSPITAL', 'COMUNA', 'REGION', 'CUT', 'COD_REGION', 'SEXO', 'EDAD', 'TRAMO_EDAD',
            'FALLECIDO', 'FECHA_INGRESO', 'FECHA_ALTA', 'MES_ALTA', 'DIAGNOSTICO1', 'DIAGNOSTICO1_DESC']

for anio in ANIOS:
    df, rep = limpiar_grd(anio)

    if FECHAS_COMO_TEXTO:
        df['FECHA_INGRESO'] = df['FECHA_INGRESO'].dt.strftime('%d/%m/%Y')

    reportes.append(rep)
    grd[anio] = df[COLS_VER].copy()
    print(f'{anio}: {rep["filas_originales"]:>7,} -> {rep["filas_finales"]:>7,} filas | '
          f'duplicados exactos eliminados: {rep["duplicados_exactos_eliminados"]}')
    del df

reporte = pd.DataFrame(reportes).set_index('anio').T
reporte

# Cantidad de valores nulos por columna en el GRD final de cada año.
nulos_por_anio = pd.DataFrame({anio: datos.isna().sum() for anio, datos in grd.items()})
nulos_por_anio

2019:  96,439 ->  96,431 filas | duplicados exactos eliminados: 0
2020:  34,962 ->  34,824 filas | duplicados exactos eliminados: 128
2021:  34,618 ->  34,616 filas | duplicados exactos eliminados: 0
2022:  66,636 ->  66,631 filas | duplicados exactos eliminados: 2
2023:  88,782 ->  88,777 filas | duplicados exactos eliminados: 0
2024:  88,764 ->  88,759 filas | duplicados exactos eliminados: 0


,2019,2020,2021,2022,2023,2024
ANIO,0,0,0,0,0,0
COD_HOSPITAL,0,0,0,0,0,0
HOSPITAL,0,0,0,0,0,0
COMUNA,0,0,0,0,0,0
REGION,41,6,0,4,1,1
CUT,41,6,0,4,1,1
COD_REGION,41,6,0,4,1,1
SEXO,0,0,0,0,0,0
EDAD,1,2,0,1,2,1
TRAMO_EDAD,0,0,0,0,0,0


## 4. Distribuciones de variables categóricas

Se muestran todos los valores distintos y su cantidad por año para `SEXO`, `COMUNA` y `MES_ALTA`.

In [12]:
for columna in ['SEXO', 'TRAMO_EDAD', 'COMUNA', 'MES_ALTA']:
    tabla = pd.concat(
        [datos[columna].value_counts(dropna=False).rename(anio) for anio, datos in grd.items()],
        axis=1,
    ).fillna(0).astype('int64').sort_index()
    if columna == 'TRAMO_EDAD':
        tabla = tabla.reindex(ETIQUETAS_TRAMO + ['DESCONOCIDO'], fill_value=0)
    print(f'\n{columna}')
    print(tabla)


SEXO
              2019   2020   2021   2022   2023   2024
SEXO                                                 
DESCONOCIDO     18      4      1      9     10      6
HOMBRE       50207  18991  19000  35486  46736  46043
MUJER        46206  15829  15615  31136  42031  42710

TRAMO_EDAD
                          2019   2020   2021   2022   2023   2024
TRAMO_EDAD                                                       
Infancia (0-4)           32806   3920   7617  23108  33555  23438
Niñez (5-12)             11982   4105   4841   9133  12517  14557
Adolescencia (13-17)      1900    759    857   1500   1686   2202
Adultos jóvenes (18-29)   3806   1954   1756   2575   3055   3399
Adultos (30-49)           6225   3615   3226   4546   5829   6687
Adultos maduros (50-64)  10001   5706   4885   6877   8591  10017
Adultos mayores (65+)    29710  14763  11434  18891  23542  28458
DESCONOCIDO                  1      2      0      1      2      1

COMUNA
                 2019  2020  2021  2022  202

## 6. Verificación rápida

Se comprueban invariantes del GRD final: fecha de ingreso válida, mes consistente, ausencia de variables descartadas y región asignada.

In [13]:
chk = pd.concat(grd.values(), ignore_index=True)
fecha_valida = chk['FECHA_ALTA'].notna().all() and chk['FECHA_INGRESO'].notna().all()
mes_consistente = chk['MES_ALTA'].equals(chk['FECHA_ALTA'].dt.month.astype('Int64'))
alta_en_su_anio = (chk['FECHA_ALTA'].dt.year == chk['ANIO']).all()
meses_completos = chk.groupby('ANIO')['MES_ALTA'].nunique().eq(12).all()
sin_auxiliares = 'FECHA_NACIMIENTO' not in chk.columns
edad_valida = chk['EDAD'].dropna().between(0, 109).all()
tramos_validos = set(chk['TRAMO_EDAD'].unique()) <= set(ETIQUETAS_TRAMO + ['DESCONOCIDO'])
tramo_consistente = (chk['EDAD'].isna() == (chk['TRAMO_EDAD'] == 'DESCONOCIDO')).all()
sin_procedimientos = not any(c.startswith('PROCEDIMIENTO') for c in chk.columns)
anios_validos = set(chk['ANIO'].unique()) == set(ANIOS)
diag_3 = chk['DIAGNOSTICO1'].str.len().eq(3).all()
codigos_ok = (chk['CUT'].notna() == chk['REGION'].notna()).all() and (chk['CUT'].dropna().str.len() == 5).all()

print(f'Fechas de alta e ingreso sin nulos: {fecha_valida}')
print(f'MES_ALTA coincide con FECHA_ALTA: {mes_consistente}')
print(f'Cada alta cae en el año de su archivo: {alta_en_su_anio}')
print(f'Los 12 meses presentes en cada año: {meses_completos}')
print(f'Sin FECHA_NACIMIENTO: {sin_auxiliares}')
print(f'EDAD entre 0 y 109: {edad_valida}')
print(f'TRAMO_EDAD solo con valores válidos: {tramos_validos}')
print(f'TRAMO_EDAD = DESCONOCIDO solo cuando falta EDAD: {tramo_consistente}')
print(f'Sin columnas PROCEDIMIENTO*: {sin_procedimientos}')
print(f'Diagnóstico principal de 3 caracteres: {diag_3}')
print(f'Solo años 2019-2024: {anios_validos}')
print(f'CUT y COD_REGION asignados donde hay región: {codigos_ok}')

region_ok = chk['REGION'].isna().mean() < 0.01
sin_region = chk.loc[chk['REGION'].isna() & (chk['COMUNA'].fillna('') != 'DESCONOCIDO'), 'COMUNA'].unique()
print(f'REGION con menos de 1% de nulos: {region_ok}')
print(f'Comunas sin región en el catálogo (salvo DESCONOCIDO): {len(sin_region)}')

assert region_ok
assert len(sin_region) == 0, sin_region

assert fecha_valida
assert mes_consistente
assert alta_en_su_anio
assert meses_completos
assert sin_auxiliares
assert edad_valida
assert tramos_validos
assert tramo_consistente
assert sin_procedimientos
assert diag_3
assert anios_validos
assert codigos_ok

print(f'\nValidacion OK: {len(chk):,} filas y {chk.shape[1]} columnas')

Fechas de alta e ingreso sin nulos: True
MES_ALTA coincide con FECHA_ALTA: True
Cada alta cae en el año de su archivo: True
Los 12 meses presentes en cada año: True
Sin FECHA_NACIMIENTO: True
EDAD entre 0 y 109: True
TRAMO_EDAD solo con valores válidos: True
TRAMO_EDAD = DESCONOCIDO solo cuando falta EDAD: True
Sin columnas PROCEDIMIENTO*: True
Diagnóstico principal de 3 caracteres: True
Solo años 2019-2024: True
CUT y COD_REGION asignados donde hay región: True
REGION con menos de 1% de nulos: True
Comunas sin región en el catálogo (salvo DESCONOCIDO): 0

Validacion OK: 410,038 filas y 16 columnas


## 6. Exportación

In [14]:
DIR_SALIDA = DIR_GRD / 'GRD_limpio'
DIR_SALIDA.mkdir(parents=True, exist_ok=True)

grd_final = pd.concat(grd.values(), ignore_index=True)
ruta_parquet = DIR_SALIDA / 'grd_respiratorio_2019_2024.parquet'
grd_final.to_parquet(ruta_parquet, index=False)

ruta_reporte = DIR_SALIDA / 'reporte_calidad.csv'
reporte.to_csv(ruta_reporte, encoding='utf-8-sig')

print(f'GRD exportado: {ruta_parquet} ({len(grd_final):,} filas, {grd_final.shape[1]} columnas)')
print(f'Reporte de calidad exportado: {ruta_reporte}')

GRD exportado: ..\GRD_CIE_TM\GRD_limpio\grd_respiratorio_2019_2024.parquet (410,038 filas, 16 columnas)
Reporte de calidad exportado: ..\GRD_CIE_TM\GRD_limpio\reporte_calidad.csv
